In [1]:
import sys, pandas as pd

In [2]:
print(sys.executable)

/Users/galinamerz/Documents/Jupyter_projects/santander-product-recommendation/.venv/bin/python


In [6]:
PATH = "/Users/galinamerz/Documents/Jupyter_projects/santander-product-recommendation/train_ver2.csv"

In [7]:
import os
print(os.getcwd())
print([f for f in os.listdir() if f.endswith(".csv")])

/Users/galinamerz/Documents/Jupyter_projects/santander-product-recommendation
['train_ver2.csv', 'sample_submission.csv', 'test_ver2.csv']


In [8]:
print(pd.__version__)

3.0.5


In [9]:
PATH = "train_ver2.csv"
ID, N_CUST, SEED = "ncodpers", 50_000, 42

# Pass 1 — customer IDs only

In [10]:
ids = pd.concat([c[ID] for c in pd.read_csv(PATH, usecols=[ID], chunksize=1_000_000)]).unique()
keep = set(pd.Series(ids).sample(N_CUST, random_state=SEED))
print(f"{len(ids):,} customers → sampling {len(keep):,}")

956,645 customers → sampling 50,000


# Pass 2 — every row for those customers

In [11]:
df = pd.concat([c[c[ID].isin(keep)] for c in pd.read_csv(PATH, chunksize=500_000)],
               ignore_index=True)

/var/folders/bs/g00sy71n7672m_f4rp6h6q680000gn/T/ipykernel_7669/2742919631.py:1: DtypeWarning: Columns (0: conyuemp) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([c[c[ID].isin(keep)] for c in pd.read_csv(PATH, chunksize=500_000)],
/var/folders/bs/g00sy71n7672m_f4rp6h6q680000gn/T/ipykernel_7669/2742919631.py:1: DtypeWarning: Columns (0: conyuemp) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([c[c[ID].isin(keep)] for c in pd.read_csv(PATH, chunksize=500_000)],
/var/folders/bs/g00sy71n7672m_f4rp6h6q680000gn/T/ipykernel_7669/2742919631.py:1: DtypeWarning: Columns (0: conyuemp) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([c[c[ID].isin(keep)] for c in pd.read_csv(PATH, chunksize=500_000)],
/var/folders/bs/g00sy71n7672m_f4rp6h6q680000gn/T/ipykernel_7669/2742919631.py:1: DtypeWarning: Columns (0: conyuemp) have mixed types. Specify dtype option on import or 

In [13]:
import numpy as np

# Numerics that arrive as padded strings

In [14]:
for col in ["age", "antiguedad", "renta"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
df.loc[df["antiguedad"] < 0, "antiguedad"] = np.nan   # new-customer sentinel

# Mixed-type object columns break the Parquet write

In [16]:
for c in [c for c in df.columns if df[c].dtype == "object"]:
    df[c] = df[c].astype(str).str.strip().replace({"nan": None, "": None})


In [17]:
df.to_parquet("santander_sample.parquet", index=False)
print(df.shape)
print(df["fecha_dato"].value_counts().sort_index())

(713861, 48)
fecha_dato
2015-01-28    32787
2015-02-28    32904
2015-03-28    33005
2015-04-28    33097
2015-05-28    33173
2015-06-28    33193
2015-07-28    43435
2015-08-28    44103
2015-09-28    45261
2015-10-28    46586
2015-11-28    47353
2015-12-28    47652
2016-01-28    47842
2016-02-28    48072
2016-03-28    48287
2016-04-28    48471
2016-05-28    48640
Name: count, dtype: int64


In [18]:
print(list(df.columns))

['fecha_dato', 'ncodpers', 'ind_empleado', 'pais_residencia', 'sexo', 'age', 'fecha_alta', 'ind_nuevo', 'antiguedad', 'indrel', 'ult_fec_cli_1t', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext', 'conyuemp', 'canal_entrada', 'indfall', 'tipodom', 'cod_prov', 'nomprov', 'ind_actividad_cliente', 'renta', 'segmento', 'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1']


Here's the label-building cell. It reshapes to long format, which makes "held last month, holds this month" a straightforward comparison rather than 24 separate column operations:

In [25]:
labels.to_parquet("labels.parquet", index=False)

In [26]:
PRODUCTS = [
    "ind_cco_fin_ult1",   # current account
    "ind_ahor_fin_ult1",  # savings account
    "ind_ctju_fin_ult1",  # junior account
    "ind_ctop_fin_ult1",  # particular account
    "ind_ctpp_fin_ult1",  # particular plus account
    "ind_deco_fin_ult1",  # short-term deposit
    "ind_deme_fin_ult1",  # medium-term deposit
    "ind_dela_fin_ult1",  # long-term deposit
    "ind_ecue_fin_ult1",  # e-account
    "ind_fond_fin_ult1",  # funds
    "ind_valo_fin_ult1",  # securities
    "ind_nomina_ult1",    # payroll
    "ind_nom_pens_ult1",  # pensions
    "ind_recibo_ult1",    # direct debit
]

In [27]:
d = df.copy()
d["month"] = pd.to_datetime(d["fecha_dato"]).dt.to_period("M")
d = d.sort_values(["ncodpers", "month"])

In [28]:
long = d.melt(
    id_vars=["ncodpers", "month"],
    value_vars=PRODUCTS,
    var_name="product",
    value_name="held",
)
long["held"] = pd.to_numeric(long["held"], errors="coerce").fillna(0).astype(int)
long = long.sort_values(["ncodpers", "product", "month"])

In [29]:
g = long.groupby(["ncodpers", "product"], sort=False)
long["held_prev"] = g["held"].shift(1)
long["prev_month"] = g["month"].shift(1)

# Only consecutive months count; gaps and first appearances can't produce a label

In [30]:
consecutive = long["month"] == long["prev_month"] + 1
long["added"] = ((long["held"] == 1) & (long["held_prev"] == 0) & consecutive).astype(int)
labels = long[consecutive].copy()

In [31]:
print(f"labelable rows: {len(labels):,}")
print(f"adoption events: {labels['added'].sum():,}  "
      f"({labels['added'].mean()*100:.3f}%)")
print(labels.groupby("product")["added"].sum().sort_values(ascending=False))

labelable rows: 9,288,608
adoption events: 23,044  (0.248%)
product
ind_recibo_ult1      8107
ind_nom_pens_ult1    4481
ind_nomina_ult1      3900
ind_cco_fin_ult1     3590
ind_ecue_fin_ult1    1373
ind_dela_fin_ult1     650
ind_valo_fin_ult1     238
ind_ctop_fin_ult1     198
ind_fond_fin_ult1     189
ind_deco_fin_ult1     156
ind_ctpp_fin_ult1     128
ind_ctju_fin_ult1      24
ind_deme_fin_ult1      10
ind_ahor_fin_ult1       0
Name: added, dtype: int64
